In [0]:
use globalretail.gr_silver 

**Customer table**

In [0]:
drop table if exists globalretail.gr_silver.customer;
create table if not exists globalretail.gr_silver.customer
as
select 
customer_id,
name as cust_name,
email as cust_email,
country ,
customer_type ,
registration_date ,
age ,
gender ,
total_purchases,
current_timestamp() AS ingestion_date
from globalretail.gr_bronze.customer

In [0]:
select * from globalretail.gr_silver.customer

- check email is valid
- valid age between 18 to 100
- remove junk records where total purchase is negative in system
- if total purchase > 1000 high value , >500 medium else low

- Apply incremental load

In [0]:
MERGE INTO globalretail.gr_silver.customer AS target
USING globalretail.gr_bronze.customer AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN 
  UPDATE SET
    target.cust_name = source.name,
    target.cust_email = source.email,
    target.country = source.country,
    target.customer_type = source.customer_type,
    target.registration_date = source.registration_date,
    target.age = source.age,
    target.gender = source.gender,
    target.total_purchases = source.total_purchases,
    target.ingestion_date = current_timestamp()

WHEN NOT MATCHED THEN 
  INSERT (
    customer_id,
    cust_name,
    cust_email,
    country,
    customer_type,
    registration_date,
    age,
    gender,
    total_purchases,
    ingestion_date
  )
  VALUES (
    source.customer_id,
    source.name,
    source.email,
    source.country,
    source.customer_type,
    source.registration_date,
    source.age,
    source.gender,
    source.total_purchases,
    current_timestamp()
  );

In [0]:
CREATE OR REPLACE TABLE globalretail.gr_silver.customer_transform
USING DELTA
AS
SELECT
    customer_id,
    cust_name,
    TRIM(cust_email) AS cust_email,
    country,
    customer_type,
    registration_date,
    CAST(age AS INT) AS age,
    gender,
    CAST(total_purchases AS INT) AS total_purchases,

    -- Customer segmentation
    CASE 
        WHEN CAST(total_purchases AS INT) > 1000 THEN 'High Value'
        WHEN CAST(total_purchases AS INT) > 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment,

    current_timestamp() AS ingestion_date

FROM globalretail.gr_silver.customer

WHERE 
    -- ✅ Email validation (robust but not too strict)
    TRIM(cust_email) LIKE '%@%.%'

    -- ✅ Age validation
    AND CAST(age AS INT) BETWEEN 18 AND 100

    -- ✅ Remove junk records
    AND CAST(total_purchases AS INT) >= 0;

In [0]:
select * from globalretail.gr_silver.customer_transform